In [ ]:
# ============================================================
#  EEG FULL PIPELINE  —  ALL STAGES  —  ds006036 (AD/FTD/CN)
#  Stage 1  : CNN-BiLSTM temporal classifier  (UNCHANGED)
#  Stage 2  : Functional connectivity (coherence matrices)
#  Stage 3  : GCN graph learning
#  Stage 4  : Connectivity entropy biomarker
#  Stage 5  : Temporal–spatial coupling (CCA)
#  Stage 6  : Early-warning / lead-time analysis
#  Run cells top-to-bottom in Kaggle / Jupyter notebook
# ============================================================


# ╔═════════════════════════════════════════════════════════════╗
# ║                        STAGE  1                            ║
# ║          CNN-BiLSTM TEMPORAL CLASSIFIER  (UNCHANGED)       ║
# ╚═════════════════════════════════════════════════════════════╝

# ─────────────────────────────────────────────────────────────
# CELL 1 — Install dependencies (run once)
# ─────────────────────────────────────────────────────────────
# !pip install mne scipy scikit-learn matplotlib seaborn tensorflow networkx -q


# ─────────────────────────────────────────────────────────────
# CELL 2 — Imports
# ─────────────────────────────────────────────────────────────
import os, glob, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.signal import welch

import mne
mne.set_log_level('WARNING')

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, balanced_accuracy_score
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, regularizers

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices('GPU'))

# Enable mixed precision for RTX GPU (faster, same accuracy)
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for g in gpus:
        tf.config.experimental.set_memory_growth(g, True)
    tf.keras.mixed_precision.set_global_policy('mixed_float16')
    print("Mixed precision ON")


# ─────────────────────────────────────────────────────────────
# CELL 3 — SET YOUR DATASET PATH HERE
# ─────────────────────────────────────────────────────────────
DATA_DIR = "/kaggle/input/datasets/shivasubrahmanyakc/new-dataset/New Dataset"
USE_PREPROCESSED = True    # True = use derivatives/eeglab/ (recommended, ICA done)
                            # False = use raw sub-XXX/eeg/ files

CACHE_FILE = "./eeg_cache.npz"   # saves processed data so 2nd run is instant
OUT_DIR    = "./results"
os.makedirs(OUT_DIR, exist_ok=True)

# Verify path
assert os.path.isdir(DATA_DIR), f"❌ Folder not found: {DATA_DIR}"
assert os.path.exists(os.path.join(DATA_DIR, "participants.tsv")), \
    "❌ participants.tsv not found inside DATA_DIR"
print("✅ Dataset folder found:", DATA_DIR)


# ─────────────────────────────────────────────────────────────
# CELL 4 — Constants (matched to ds006036 specs)
# ─────────────────────────────────────────────────────────────
CHANNELS = ['Fp1','Fp2','F7','F3','Fz','F4','F8',
            'T3','C3','Cz','C4','T4',
            'T5','P3','Pz','P4','T6',
            'O1','O2']

GROUP_MAP = {'A':'AD', 'F':'FTD', 'C':'CN',
             'AD':'AD', 'FTD':'FTD', 'CN':'CN'}

CLASS_NAMES  = ['AD', 'CN', 'FTD']
NATIVE_SFREQ = 500
TARGET_SFREQ = 256
SEG_SEC      = 4
OVERLAP      = 0.5
AMP_THRESH   = 150.0
N_TIMES      = SEG_SEC * TARGET_SFREQ   # 1024 samples
N_CH         = 19
N_CLASSES    = 3

BANDS = {
    'delta': (0.5,  4.0),
    'theta': (4.0,  8.0),
    'alpha': (8.0, 13.0),
    'beta' : (13.0, 30.0),
    'gamma': (30.0, 45.0),
}
N_BANDS = len(BANDS)
print("Constants set ✅")


# ─────────────────────────────────────────────────────────────
# CELL 5 — Load participant labels
# ─────────────────────────────────────────────────────────────
def load_labels(data_dir):
    df = pd.read_csv(os.path.join(data_dir, 'participants.tsv'), sep='\t')
    df.columns = [c.strip().lower() for c in df.columns]
    gcol = next((c for c in df.columns
                 if c in ['group','diagnosis','label','class']), None)
    assert gcol, f"No group column found. Columns: {df.columns.tolist()}"
    labels = {}
    for _, row in df.iterrows():
        pid   = str(row['participant_id']).strip()
        grp   = str(row[gcol]).strip().upper()
        label = GROUP_MAP.get(grp)
        if label:
            labels[pid] = label
    return labels

labels = load_labels(DATA_DIR)
for cls in CLASS_NAMES:
    print(f"  {cls}: {sum(v==cls for v in labels.values())} subjects")


# ─────────────────────────────────────────────────────────────
# CELL 6 — Find .set file for each subject
# ─────────────────────────────────────────────────────────────
def find_set(data_dir, sub_id, use_preprocessed):
    if use_preprocessed:
        pat = os.path.join(data_dir, 'derivatives', 'eeglab',
                           sub_id, 'eeg', '*.set')
        hits = glob.glob(pat)
        if hits:
            return hits[0], True
    hits = glob.glob(os.path.join(data_dir, sub_id, 'eeg', '*.set'))
    return (hits[0], False) if hits else (None, False)


# ─────────────────────────────────────────────────────────────
# CELL 7 — Preprocess one subject
# ─────────────────────────────────────────────────────────────
def preprocess_subject(set_path, is_preprocessed):
    raw = mne.io.read_raw_eeglab(set_path, preload=True, verbose=False)
    ch_map = {c.upper(): c for c in raw.ch_names}
    keep   = [ch_map[c.upper()] for c in CHANNELS if c.upper() in ch_map]
    assert len(keep) >= 15, f"Only {len(keep)} channels found in {set_path}"
    raw.pick_channels(keep, ordered=False)
    raw.reorder_channels([ch_map[c.upper()] for c in CHANNELS
                          if c.upper() in ch_map])
    if not is_preprocessed:
        raw.notch_filter(50.0, verbose=False)
        raw.filter(0.5, 45.0, verbose=False)
        raw.set_eeg_reference('average', projection=False, verbose=False)
    if raw.info['sfreq'] != TARGET_SFREQ:
        raw.resample(TARGET_SFREQ, verbose=False)
    return raw


# ─────────────────────────────────────────────────────────────
# CELL 8 — Segment signal into fixed-length windows
# ─────────────────────────────────────────────────────────────
def segment(raw):
    data = raw.get_data() * 1e6
    step = int(N_TIMES * (1 - OVERLAP))
    segs = []
    for s in range(0, data.shape[1] - N_TIMES + 1, step):
        w = data[:, s:s + N_TIMES]
        if np.max(np.abs(w)) <= AMP_THRESH:
            segs.append(w.astype(np.float32))
    return np.array(segs) if segs else None   # (N, 19, 1024)


# ─────────────────────────────────────────────────────────────
# CELL 9 — Band-power features
# ─────────────────────────────────────────────────────────────
def band_powers(segs):
    N, nch, nt = segs.shape
    out = np.zeros((N, nch, N_BANDS), dtype=np.float32)
    for i in range(N):
        for c in range(nch):
            freqs, psd = welch(segs[i, c], fs=TARGET_SFREQ,
                               nperseg=TARGET_SFREQ,
                               noverlap=TARGET_SFREQ // 2)
            for b, (lo, hi) in enumerate(BANDS.values()):
                mask = (freqs >= lo) & (freqs < hi)
                pwr  = np.trapz(psd[mask], freqs[mask]) if mask.any() else 1e-12
                out[i, c, b] = np.log(pwr + 1e-12)
    return out


# ─────────────────────────────────────────────────────────────
# CELL 10 — Build full dataset (or load from cache)
# ─────────────────────────────────────────────────────────────
def build_dataset():
    if os.path.exists(CACHE_FILE):
        print(f"Loading cache: {CACHE_FILE}")
        d  = np.load(CACHE_FILE, allow_pickle=True)
        le = LabelEncoder(); le.classes_ = d['class_names']
        print(f"  {len(d['y'])} segments loaded from cache")
        return d['X_raw'], d['X_band'], d['y'], le

    le = LabelEncoder(); le.fit(CLASS_NAMES)
    all_raw, all_band, all_y = [], [], []
    ok = skip = 0

    for sub_id, label_str in sorted(labels.items()):
        path, is_prep = find_set(DATA_DIR, sub_id, USE_PREPROCESSED)
        if path is None:
            print(f"  ⚠  {sub_id}: no .set file — skip"); skip += 1; continue
        try:
            raw  = preprocess_subject(path, is_prep)
            segs = segment(raw)
            if segs is None or len(segs) == 0:
                print(f"  ⚠  {sub_id}: all segments rejected"); skip += 1; continue
            bp  = band_powers(segs)
            lbl = le.transform([label_str])[0]
            all_raw.append(segs.transpose(0, 2, 1))
            all_band.append(bp)
            all_y.extend([lbl] * len(segs))
            print(f"  ✅ {sub_id} [{label_str}] {'prep' if is_prep else 'raw'}: {len(segs)} segs")
            ok += 1
        except Exception as e:
            print(f"  ❌ {sub_id}: {e}"); skip += 1

    print(f"\nLoaded {ok} subjects, skipped {skip}")
    assert all_raw, "No data loaded — check DATA_DIR and file structure"

    X_raw  = np.concatenate(all_raw,  axis=0)   # (N, 1024, 19)
    X_band = np.concatenate(all_band, axis=0)   # (N, 19, 5)
    y      = np.array(all_y, dtype=np.int32)

    np.savez_compressed(CACHE_FILE,
                        X_raw=X_raw, X_band=X_band,
                        y=y, class_names=le.classes_)
    print(f"Saved cache → {CACHE_FILE}")
    print(f"X_raw: {X_raw.shape}  X_band: {X_band.shape}  y: {y.shape}")
    for i, cn in enumerate(CLASS_NAMES):
        print(f"  {cn}: {(y==i).sum()} segments")
    return X_raw, X_band, y, le

X_raw, X_band, y, le = build_dataset()


# ─────────────────────────────────────────────────────────────
# CELL 11 — Train / val / test split + normalise
# ─────────────────────────────────────────────────────────────
def split_and_normalise(X_raw, X_band, y):
    N = len(y)
    sss1 = StratifiedShuffleSplit(1, test_size=0.15, random_state=42)
    tv_idx, test_idx = next(sss1.split(np.zeros(N), y))

    sss2 = StratifiedShuffleSplit(1, test_size=0.15/0.85, random_state=42)
    sub_tr, sub_val = next(sss2.split(np.zeros(len(tv_idx)), y[tv_idx]))
    tr_idx  = tv_idx[sub_tr]
    val_idx = tv_idx[sub_val]

    scaler = StandardScaler()
    scaler.fit(X_raw[tr_idx].reshape(-1, N_CH))

    def norm(X):
        s = X.shape
        return scaler.transform(X.reshape(-1, N_CH)).reshape(s)

    sp = dict(
        Xr_tr=norm(X_raw[tr_idx]),   Xb_tr=X_band[tr_idx],   y_tr=y[tr_idx],
        Xr_va=norm(X_raw[val_idx]),  Xb_va=X_band[val_idx],  y_va=y[val_idx],
        Xr_te=norm(X_raw[test_idx]), Xb_te=X_band[test_idx], y_te=y[test_idx],
    )
    print(f"Train: {len(tr_idx)}  Val: {len(val_idx)}  Test: {len(test_idx)}")
    return sp

sp = split_and_normalise(X_raw, X_band, y)


# ─────────────────────────────────────────────────────────────
# CELL 12 — Model: CNN-BiLSTM + Band-Power branch + Attention
# ─────────────────────────────────────────────────────────────
def build_model():
    # ── Branch A: raw EEG segments (time × channels) ─────────
    eeg_in = layers.Input(shape=(N_TIMES, N_CH), name='eeg')

    x = layers.Conv1D(64,  7, padding='same', kernel_regularizer=regularizers.l2(1e-4))(eeg_in)
    x = layers.BatchNormalization()(x); x = layers.Activation('relu')(x)
    x = layers.SpatialDropout1D(0.1)(x); x = layers.MaxPooling1D(2)(x)

    x = layers.Conv1D(128, 5, padding='same', kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x); x = layers.Activation('relu')(x)
    x = layers.SpatialDropout1D(0.1)(x); x = layers.MaxPooling1D(2)(x)

    x = layers.Conv1D(256, 3, padding='same', kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x); x = layers.Activation('relu')(x)
    x = layers.MaxPooling1D(2)(x)

    x = layers.Bidirectional(layers.LSTM(128, return_sequences=True,
                                          recurrent_dropout=0.1))(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Bidirectional(layers.LSTM(64,  return_sequences=False))(x)
    x = layers.Dropout(0.2)(x)
    temporal = x   # (batch, 128)

    # ── Branch B: band-power features (channels × bands) ─────
    band_in = layers.Input(shape=(N_CH, N_BANDS), name='band')
    b = layers.Flatten()(band_in)
    b = layers.Dense(128, activation='relu',
                     kernel_regularizer=regularizers.l2(1e-4))(b)
    b = layers.BatchNormalization()(b); b = layers.Dropout(0.2)(b)
    b = layers.Dense(64,  activation='relu')(b)
    spectral = b   # (batch, 64)

    # ── Merge ─────────────────────────────────────────────────
    merged = layers.Concatenate(name='merged')([temporal, spectral])  # (batch, 192)

    attn = layers.Dense(64,   activation='relu')(merged)
    attn = layers.Dense(N_CH, activation='softmax',
                        dtype='float32', name='channel_attn')(attn)

    x = layers.Concatenate()([merged, attn])

    # ── Classifier ────────────────────────────────────────────
    x = layers.Dense(256, kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x); x = layers.Activation('relu')(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(128, kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x); x = layers.Activation('relu')(x)
    x = layers.Dropout(0.3)(x)

    out = layers.Dense(N_CLASSES, activation='softmax',
                       dtype='float32', name='output')(x)

    model      = Model([eeg_in, band_in], out,  name='CNN_BiLSTM')
    attn_model = Model([eeg_in, band_in], attn, name='AttnExtractor')
    return model, attn_model

model, attn_model = build_model()
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3, clipnorm=1.0),
    loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=['accuracy',
             keras.metrics.AUC(name='auc'),
             keras.metrics.Precision(name='prec'),
             keras.metrics.Recall(name='rec')]
)
model.summary(line_length=85)


# ─────────────────────────────────────────────────────────────
# CELL 13 — Train
# ─────────────────────────────────────────────────────────────
os.makedirs('./checkpoints', exist_ok=True)

cw_arr = compute_class_weight('balanced', classes=np.unique(sp['y_tr']), y=sp['y_tr'])
cw     = dict(enumerate(cw_arr))
print("Class weights:", {CLASS_NAMES[k]: round(v,3) for k,v in cw.items()})

y_tr_oh = keras.utils.to_categorical(sp['y_tr'], N_CLASSES)
y_va_oh = keras.utils.to_categorical(sp['y_va'], N_CLASSES)

cbs = [
    keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=20,
                                   restore_best_weights=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                       patience=8, min_lr=1e-6, verbose=1),
    keras.callbacks.ModelCheckpoint('./checkpoints/best_model.keras',
                                     monitor='val_accuracy',
                                     save_best_only=True, verbose=1),
]

history = model.fit(
    [sp['Xr_tr'], sp['Xb_tr']], y_tr_oh,
    validation_data=([sp['Xr_va'], sp['Xb_va']], y_va_oh),
    epochs=100,
    batch_size=32,
    class_weight=cw,
    callbacks=cbs,
    verbose=1
)


# ─────────────────────────────────────────────────────────────
# CELL 14 — Evaluate on test set
# ─────────────────────────────────────────────────────────────
y_prob = model.predict([sp['Xr_te'], sp['Xb_te']], batch_size=64, verbose=0)
y_pred = np.argmax(y_prob, axis=1)
y_true = sp['y_te']

print("\n" + "="*55)
print("STAGE 1 — TEST SET RESULTS")
print("="*55)
print(classification_report(y_true, y_pred,
                             target_names=CLASS_NAMES, digits=4))
print(f"Balanced Accuracy: {balanced_accuracy_score(y_true, y_pred):.4f}")


# ─────────────────────────────────────────────────────────────
# CELL 15 — Plot: training curves
# ─────────────────────────────────────────────────────────────
fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4))
a1.plot(history.history['accuracy'],     lw=2, label='Train')
a1.plot(history.history['val_accuracy'], lw=2, label='Val')
a1.set_title('Accuracy'); a1.set_xlabel('Epoch')
a1.legend(); a1.grid(alpha=0.3)

a2.plot(history.history['loss'],     lw=2, label='Train')
a2.plot(history.history['val_loss'], lw=2, label='Val')
a2.set_title('Loss'); a2.set_xlabel('Epoch')
a2.legend(); a2.grid(alpha=0.3)

plt.suptitle('CNN-BiLSTM Training', fontsize=14)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/training_curves.png', dpi=150)
plt.show()


# ─────────────────────────────────────────────────────────────
# CELL 16 — Plot: confusion matrix
# ─────────────────────────────────────────────────────────────
cm_s1     = confusion_matrix(y_true, y_pred)
cm_pct_s1 = cm_s1.astype(float) / cm_s1.sum(axis=1, keepdims=True) * 100

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm_pct_s1, annot=True, fmt='.1f',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            cmap='Blues', vmin=0, vmax=100, ax=ax)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title('Confusion Matrix  (% per true class)')
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/confusion_matrix.png', dpi=150)
plt.show()


# ─────────────────────────────────────────────────────────────
# CELL 17 — Plot: mean band power per class
# ─────────────────────────────────────────────────────────────
band_names = list(BANDS.keys())
colors     = ['#E74C3C', '#2ECC71', '#3498DB']

fig, ax = plt.subplots(figsize=(8, 4))
for cls in range(N_CLASSES):
    mask    = y == cls
    mean_bp = X_band[mask].mean(axis=1).mean(axis=0)
    std_bp  = X_band[mask].mean(axis=1).std(axis=0)
    ax.plot(band_names, mean_bp, color=colors[cls], lw=2.5,
            marker='o', label=CLASS_NAMES[cls])
    ax.fill_between(range(5), mean_bp - std_bp,
                    mean_bp + std_bp, color=colors[cls], alpha=0.15)

ax.set_title('Mean Log Band Power by Class\n(Delta↑ + Alpha↓ = hallmark of AD)')
ax.set_xlabel('Frequency Band'); ax.set_ylabel('Log Power (µV²/Hz)')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/band_power.png', dpi=150)
plt.show()


# ─────────────────────────────────────────────────────────────
# CELL 18 — Plot: channel attention per class
# ─────────────────────────────────────────────────────────────
attn_w = attn_model.predict([sp['Xr_te'], sp['Xb_te']],
                              batch_size=64, verbose=0)

fig, axes = plt.subplots(1, N_CLASSES, figsize=(6*N_CLASSES, 4))
for cls, ax in enumerate(axes):
    mask = y_true == cls
    if not mask.any(): continue
    mean_w = attn_w[mask].mean(axis=0)
    bars   = ax.bar(CHANNELS, mean_w,
                    color=colors[cls], alpha=0.8, edgecolor='black')
    top3   = np.argsort(mean_w)[-3:]
    for i in top3:
        bars[i].set_edgecolor('goldenrod'); bars[i].set_linewidth(2.5)
    ax.set_title(f'{CLASS_NAMES[cls]} — Channel Attention')
    ax.set_xlabel('Electrode'); ax.tick_params(axis='x', rotation=45)
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('Which electrodes the model attends to  (gold = top-3)', fontsize=12)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/channel_attention.png', dpi=150)
plt.show()

print("\nStage 1 complete ✅")
print("All plots saved to:", OUT_DIR)
print("Best model saved to: ./checkpoints/best_model.keras")

# ╔═════════════════════════════════════════════════════════════╗
# ║                        STAGE  2                            ║
# ║       FUNCTIONAL CONNECTIVITY CONSTRUCTION                 ║
# ║  Coherence-based (19×19) adjacency matrix per segment      ║
# ╚═════════════════════════════════════════════════════════════╝

# ─────────────────────────────────────────────────────────────
# CELL 19 — Additional imports for Stages 2–6
# ─────────────────────────────────────────────────────────────
import networkx as nx
from scipy.signal import coherence
from scipy.stats import entropy as scipy_entropy
from scipy.stats import kruskal, mannwhitneyu
from sklearn.cross_decomposition import CCA
from collections import Counter

print("Stages 2–6 imports done ✅")


# ─────────────────────────────────────────────────────────────
# CELL 20 — Coherence-based adjacency matrix per segment
# ─────────────────────────────────────────────────────────────
def compute_coherence_matrix(seg, fs=TARGET_SFREQ, lo=0.5, hi=45.0):
    """
    seg  : (N_times, N_ch)  — one EEG segment, time-first
    Returns (N_ch, N_ch) float32 mean-coherence adjacency matrix.
    """
    nch = seg.shape[1]
    A   = np.zeros((nch, nch), dtype=np.float32)
    for i in range(nch):
        for j in range(i + 1, nch):
            f, Cxy = coherence(seg[:, i], seg[:, j], fs=fs, nperseg=fs)
            mask   = (f >= lo) & (f < hi)
            coh    = float(Cxy[mask].mean()) if mask.any() else 0.0
            A[i, j] = coh
            A[j, i] = coh
    np.fill_diagonal(A, 1.0)   # self-loops
    return A


def build_connectivity_dataset(X_raw_norm, y_labels,
                                cache='./conn_cache.npz'):
    """
    X_raw_norm : (N, T, CH) — raw segments (time-first)
    Returns A_all (N, CH, CH), y_labels (N,)
    """
    if os.path.exists(cache):
        print(f"Loading connectivity cache: {cache}")
        d = np.load(cache)
        print(f"  {len(d['y'])} adjacency matrices loaded")
        return d['A_all'], d['y']

    N     = len(y_labels)
    A_all = np.zeros((N, N_CH, N_CH), dtype=np.float32)
    print(f"Computing coherence matrices for {N} segments …")
    for i in range(N):
        if i % 500 == 0:
            print(f"  {i}/{N}")
        A_all[i] = compute_coherence_matrix(X_raw_norm[i])

    np.savez_compressed(cache, A_all=A_all, y=y_labels)
    print(f"Saved → {cache}")
    return A_all, y_labels


# X_raw shape is (N, 1024, 19) — already time-first, perfect for coherence
A_all, y_conn = build_connectivity_dataset(X_raw, y, cache='./conn_cache.npz')
print(f"Adjacency dataset: {A_all.shape}  labels: {y_conn.shape}")


# ─────────────────────────────────────────────────────────────
# CELL 21 — Visualise mean connectivity matrix per class
# ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, N_CLASSES, figsize=(5 * N_CLASSES, 4))
for cls, ax in enumerate(axes):
    mask   = y_conn == cls
    mean_A = A_all[mask].mean(axis=0).copy()
    np.fill_diagonal(mean_A, 0)
    im = ax.imshow(mean_A, cmap='hot', vmin=0, vmax=1)
    ax.set_title(f'{CLASS_NAMES[cls]}\nMean Coherence')
    ax.set_xticks(range(N_CH)); ax.set_xticklabels(CHANNELS, rotation=90, fontsize=6)
    ax.set_yticks(range(N_CH)); ax.set_yticklabels(CHANNELS, fontsize=6)
    plt.colorbar(im, ax=ax, fraction=0.046)

plt.suptitle('Stage 2 — Functional Connectivity (mean coherence per class)', fontsize=12)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/stage2_connectivity_matrices.png', dpi=150)
plt.show()
print("Stage 2 complete ✅")


# ╔═════════════════════════════════════════════════════════════╗
# ║                        STAGE  3                            ║
# ║              GRAPH LEARNING WITH GCN                       ║
# ║  Node features: band-power (19 × 5)                        ║
# ║  Edges        : coherence adjacency from Stage 2           ║
# ╚═════════════════════════════════════════════════════════════╝

# ─────────────────────────────────────────────────────────────
# CELL 22 — Prepare node features and graph splits
# ─────────────────────────────────────────────────────────────
N_graph = len(y_conn)
X_node  = X_band[:N_graph]    # (N, 19, 5)
A_graph = A_all               # (N, 19, 19)
y_graph = y_conn              # (N,)

sss1 = StratifiedShuffleSplit(1, test_size=0.15, random_state=42)
tv_idx_g, te_idx_g = next(sss1.split(np.zeros(N_graph), y_graph))
sss2 = StratifiedShuffleSplit(1, test_size=0.15/0.85, random_state=42)
sub_tr_g, sub_val_g = next(sss2.split(np.zeros(len(tv_idx_g)), y_graph[tv_idx_g]))
tr_idx_g  = tv_idx_g[sub_tr_g]
val_idx_g = tv_idx_g[sub_val_g]

print(f"GCN — Train: {len(tr_idx_g)}  Val: {len(val_idx_g)}  Test: {len(te_idx_g)}")


# ─────────────────────────────────────────────────────────────
# CELL 23 — Manual GCN layer (no extra dependencies needed)
# ─────────────────────────────────────────────────────────────
class GCNLayer(layers.Layer):
    """
    H' = σ( D^{-1/2} A D^{-1/2} H W + b )
    Inputs: [H (B, N, F),  A (B, N, N)]
    """
    def __init__(self, units, activation='relu', **kw):
        super().__init__(**kw)
        self.units = units
        self.act   = layers.Activation(activation) if activation else None

    def build(self, input_shape):
        F = input_shape[0][-1]
        self.W = self.add_weight(name='W', shape=(F, self.units),
                                 initializer='glorot_uniform',
                                 regularizer=regularizers.l2(1e-4))
        self.b = self.add_weight(name='b', shape=(self.units,),
                                 initializer='zeros')
        super().build(input_shape)

    def call(self, inputs):
        H, A = inputs
        deg      = tf.reduce_sum(A, axis=-1) + 1e-8         # (B, N)
        d_inv_sq = tf.linalg.diag(tf.pow(deg, -0.5))        # (B, N, N)
        A_hat    = d_inv_sq @ A @ d_inv_sq                   # (B, N, N)
        out      = A_hat @ H @ self.W + self.b               # (B, N, units)
        return self.act(out) if self.act else out


# ─────────────────────────────────────────────────────────────
# CELL 24 — Build GCN model
# ─────────────────────────────────────────────────────────────
def build_gcn():
    node_in = layers.Input(shape=(N_CH, N_BANDS), name='node_feat')
    adj_in  = layers.Input(shape=(N_CH, N_CH),    name='adj')

    h = GCNLayer(64,  activation='relu',  name='gcn1')([node_in, adj_in])
    h = layers.BatchNormalization()(h); h = layers.Dropout(0.3)(h)

    h = GCNLayer(128, activation='relu',  name='gcn2')([h, adj_in])
    h = layers.BatchNormalization()(h); h = layers.Dropout(0.3)(h)

    h = GCNLayer(64,  activation='relu',  name='gcn3')([h, adj_in])

    # Global readout: mean + max pooling → graph-level embedding
    mean_pool = layers.GlobalAveragePooling1D()(h)   # (B, 64)
    max_pool  = layers.GlobalMaxPooling1D()(h)        # (B, 64)
    graph_emb = layers.Concatenate(name='graph_emb')([mean_pool, max_pool])  # (B, 128)

    x = layers.Dense(128, kernel_regularizer=regularizers.l2(1e-4))(graph_emb)
    x = layers.BatchNormalization()(x); x = layers.Activation('relu')(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(64)(x)
    x = layers.BatchNormalization()(x); x = layers.Activation('relu')(x)
    x = layers.Dropout(0.3)(x)

    out = layers.Dense(N_CLASSES, activation='softmax',
                       dtype='float32', name='gcn_output')(x)

    gcn_clf = Model([node_in, adj_in], out,       name='GCN_Clf')
    gcn_emb = Model([node_in, adj_in], graph_emb, name='GCN_Emb')
    return gcn_clf, gcn_emb

gcn_model, gcn_embed_model = build_gcn()
gcn_model.compile(
    optimizer=keras.optimizers.Adam(1e-3, clipnorm=1.0),
    loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=['accuracy', keras.metrics.AUC(name='auc')]
)
gcn_model.summary(line_length=80)


# ─────────────────────────────────────────────────────────────
# CELL 25 — Train GCN
# ─────────────────────────────────────────────────────────────
cw_arr_g = compute_class_weight('balanced',
                                 classes=np.unique(y_graph[tr_idx_g]),
                                 y=y_graph[tr_idx_g])
cw_gcn = dict(enumerate(cw_arr_g))
print("GCN class weights:", {CLASS_NAMES[k]: round(v,3) for k,v in cw_gcn.items()})

y_tr_oh_g = keras.utils.to_categorical(y_graph[tr_idx_g],  N_CLASSES)
y_va_oh_g = keras.utils.to_categorical(y_graph[val_idx_g], N_CLASSES)

gcn_cbs = [
    keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=20,
                                   restore_best_weights=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                       patience=8, min_lr=1e-6, verbose=1),
    keras.callbacks.ModelCheckpoint('./checkpoints/best_gcn.keras',
                                     monitor='val_accuracy',
                                     save_best_only=True, verbose=1),
]

gcn_history = gcn_model.fit(
    [X_node[tr_idx_g],  A_graph[tr_idx_g]],  y_tr_oh_g,
    validation_data=(
        [X_node[val_idx_g], A_graph[val_idx_g]], y_va_oh_g),
    epochs=100,
    batch_size=32,
    class_weight=cw_gcn,
    callbacks=gcn_cbs,
    verbose=1
)


# ─────────────────────────────────────────────────────────────
# CELL 26 — Evaluate GCN on test set
# ─────────────────────────────────────────────────────────────
y_prob_g = gcn_model.predict(
    [X_node[te_idx_g], A_graph[te_idx_g]], batch_size=64, verbose=0)
y_pred_g = np.argmax(y_prob_g, axis=1)
y_true_g = y_graph[te_idx_g]

print("\n" + "="*55)
print("STAGE 3 — GCN TEST SET RESULTS")
print("="*55)
print(classification_report(y_true_g, y_pred_g,
                             target_names=CLASS_NAMES, digits=4))
print(f"Balanced Accuracy: {balanced_accuracy_score(y_true_g, y_pred_g):.4f}")


# ─────────────────────────────────────────────────────────────
# CELL 27 — Plot GCN training curves + confusion matrix
# ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(gcn_history.history['loss'],     lw=2, label='Train')
axes[0].plot(gcn_history.history['val_loss'], lw=2, label='Val')
axes[0].set_title('GCN Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(gcn_history.history['accuracy'],     lw=2, label='Train')
axes[1].plot(gcn_history.history['val_accuracy'], lw=2, label='Val')
axes[1].set_title('GCN Accuracy'); axes[1].legend(); axes[1].grid(alpha=0.3)

cm_g     = confusion_matrix(y_true_g, y_pred_g)
cm_pct_g = cm_g / cm_g.sum(axis=1, keepdims=True) * 100
sns.heatmap(cm_pct_g, annot=True, fmt='.1f',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            cmap='Greens', vmin=0, vmax=100, ax=axes[2])
axes[2].set_xlabel('Predicted'); axes[2].set_ylabel('True')
axes[2].set_title('GCN Confusion Matrix (%)')

plt.suptitle('Stage 3 — GCN Results', fontsize=13)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/stage3_gcn_results.png', dpi=150)
plt.show()
print("Stage 3 complete ✅")


# ╔═════════════════════════════════════════════════════════════╗
# ║                        STAGE  4                            ║
# ║           CONNECTIVITY ENTROPY BIOMARKER                   ║
# ║  Measures network disorder: high entropy = dementia        ║
# ╚═════════════════════════════════════════════════════════════╝

# ─────────────────────────────────────────────────────────────
# CELL 28 — Compute entropy + graph metrics per segment
# ─────────────────────────────────────────────────────────────
def graph_entropy_features(A):
    """
    A : (CH, CH) adjacency matrix (coherence weights, with self-loops)
    Returns dict of discriminative graph biomarkers.
    """
    A_nsl = A.copy(); np.fill_diagonal(A_nsl, 0)

    # 1. Von Neumann entropy (Laplacian eigenspectrum)
    deg     = A_nsl.sum(axis=1)
    D_mat   = np.diag(deg)
    L       = D_mat - A_nsl
    D_inv   = np.diag(1.0 / (deg + 1e-12))
    L_rw    = D_inv @ L
    eigvals = np.abs(np.linalg.eigvals(L_rw)).real
    eigvals = eigvals / (eigvals.sum() + 1e-12)
    vne     = float(scipy_entropy(eigvals + 1e-12))

    # 2. Edge-weight entropy — shift off zero floor so near-uniform
    #    matrices still produce variance across segments
    tri     = A_nsl[np.triu_indices(N_CH, k=1)]
    tri_pos = tri - tri.min() + 1e-6
    edge_ent = float(scipy_entropy(tri_pos / tri_pos.sum()))

    # 3. Node-strength entropy
    str_norm     = deg / (deg.sum() + 1e-12)
    strength_ent = float(scipy_entropy(str_norm + 1e-12))

    # 4. Mean coherence (overall connectivity level)
    mean_coh = float(tri.mean())

    # 5. Variance of edge weights (heterogeneity)
    var_coh  = float(tri.var())

    # 6. Global efficiency
    G          = nx.from_numpy_array(A_nsl)
    global_eff = float(nx.global_efficiency(G))

    # 7. Clustering coefficient — adaptive threshold = median
    thresh = float(np.median(tri[tri > 0])) if (tri > 0).any() else 0.3
    G_thr  = nx.from_numpy_array((A_nsl > thresh).astype(float))
    cluster = float(nx.average_clustering(G_thr))

    return {
        'vne':          vne,
        'edge_entropy': edge_ent,
        'strength_ent': strength_ent,
        'mean_coh':     mean_coh,
        'var_coh':      var_coh,
        'global_eff':   global_eff,
        'clustering':   cluster,
    }


def compute_entropy_dataset(A_all, y_labels):
    records = []
    print(f"Computing entropy features for {len(A_all)} graphs …")
    for i, (A, lbl) in enumerate(zip(A_all, y_labels)):
        if i % 500 == 0: print(f"  {i}/{len(A_all)}")
        feat = graph_entropy_features(A)
        feat['label'] = CLASS_NAMES[lbl]
        records.append(feat)
    return pd.DataFrame(records)

entropy_df = compute_entropy_dataset(A_all, y_conn)
print("\nMean entropy metrics per class:")
print(entropy_df.groupby('label').mean().round(4))


# ─────────────────────────────────────────────────────────────
# CELL 29 — Plot entropy biomarkers by class
# ─────────────────────────────────────────────────────────────
entropy_metrics = ['vne','edge_entropy','strength_ent',
                   'mean_coh','var_coh','global_eff','clustering']
palette = {'AD':'#E74C3C','CN':'#2ECC71','FTD':'#3498DB'}

# Print quick stats to verify variance exists
print("\nEntropy metric variance across all segments:")
for m in entropy_metrics:
    print(f"  {m:<18}  std={entropy_df[m].std():.6f}  "
          f"min={entropy_df[m].min():.4f}  max={entropy_df[m].max():.4f}")

# Plot — two rows of subplots so labels don't crowd
n_m  = len(entropy_metrics)
ncol = 4
nrow = (n_m + ncol - 1) // ncol
fig, axes = plt.subplots(nrow, ncol, figsize=(5*ncol, 4*nrow))
axes_flat = axes.flatten() if nrow > 1 else axes
for ax, metric in zip(axes_flat, entropy_metrics):
    sns.boxplot(data=entropy_df, x='label', y=metric,
                palette=palette, order=CLASS_NAMES, ax=ax,
                width=0.5, linewidth=1.2)
    ax.set_title(metric.replace('_',' ').title())
    ax.set_xlabel(''); ax.grid(axis='y', alpha=0.3)
for ax in axes_flat[n_m:]:          # hide unused subplots
    ax.set_visible(False)

plt.suptitle('Stage 4 — Connectivity Biomarkers\n'
             'AD/FTD show higher entropy / lower efficiency', fontsize=12)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/stage4_entropy_biomarkers.png', dpi=150)
plt.show()


# ─────────────────────────────────────────────────────────────
# CELL 30 — Statistical significance (safe wrapper)
# ─────────────────────────────────────────────────────────────
def safe_kruskal(*groups):
    """Kruskal-Wallis with guard against all-identical values."""
    # Filter out groups with zero variance
    valid = [g for g in groups if g.std() > 0]
    if len(valid) < 2:
        return np.nan, np.nan
    # Check combined variance
    combined = np.concatenate(valid)
    if combined.std() == 0:
        return np.nan, np.nan
    try:
        return kruskal(*valid)
    except ValueError:
        return np.nan, np.nan

def safe_mannwhitney(a, b, alternative='greater'):
    """Mann-Whitney U with guard against identical arrays."""
    if a.std() == 0 and b.std() == 0:
        return np.nan, np.nan
    try:
        u, p = mannwhitneyu(a, b, alternative=alternative)
        return u, p
    except ValueError:
        return np.nan, np.nan

print("\nKruskal-Wallis test (AD vs FTD vs CN):")
for metric in entropy_metrics:
    groups = [entropy_df[entropy_df['label']==c][metric].values
              for c in CLASS_NAMES]
    stat, p = safe_kruskal(*groups)
    if np.isnan(stat):
        print(f"  {metric:<18}  skipped (no variance)")
        continue
    sig = '***' if p<0.001 else ('**' if p<0.01 else ('*' if p<0.05 else 'ns'))
    print(f"  {metric:<18}  H={stat:7.2f}   p={p:.4e}  {sig}")

print("\nMann-Whitney U: AD vs CN (greater = AD has higher value):")
for metric in entropy_metrics:
    ad = entropy_df[entropy_df['label']=='AD'][metric].values
    cn = entropy_df[entropy_df['label']=='CN'][metric].values
    u, p = safe_mannwhitney(ad, cn, alternative='greater')
    if np.isnan(u):
        print(f"  {metric:<18}  skipped (no variance)")
        continue
    sig = '***' if p<0.001 else ('**' if p<0.01 else ('*' if p<0.05 else 'ns'))
    print(f"  {metric:<18}  U={u:8.0f}  p={p:.4e}  {sig}")

entropy_df.to_csv(f'{OUT_DIR}/entropy_biomarkers.csv', index=False)
print(f"\nSaved → {OUT_DIR}/entropy_biomarkers.csv")
print("Stage 4 complete ✅")


# ╔═════════════════════════════════════════════════════════════╗
# ║                        STAGE  5                            ║
# ║         TEMPORAL–SPATIAL COUPLING ANALYSIS                 ║
# ║  CCA between Stage-1 temporal embeddings                   ║
# ║  and Stage-3 graph embeddings                              ║
# ╚═════════════════════════════════════════════════════════════╝

# ─────────────────────────────────────────────────────────────
# CELL 31 — Extract temporal + graph embeddings
# ─────────────────────────────────────────────────────────────
# Temporal embedding: the 192-dim 'merged' vector from CNN-BiLSTM
temporal_extractor = Model(
    inputs  = model.input,
    outputs = model.get_layer('merged').output,
    name    = 'TemporalExtractor'
)

print("Extracting temporal embeddings (Stage 1) …")
temp_emb_all = temporal_extractor.predict(
    [X_raw[:N_graph], X_band[:N_graph]],
    batch_size=64, verbose=1
)                                          # (N_graph, 192)

print("Extracting graph embeddings (Stage 3) …")
graph_emb_all = gcn_embed_model.predict(
    [X_node, A_graph],
    batch_size=64, verbose=1
)                                          # (N_graph, 128)

print(f"Temporal embeddings : {temp_emb_all.shape}")
print(f"Graph embeddings    : {graph_emb_all.shape}")


# ─────────────────────────────────────────────────────────────
# CELL 32 — Canonical Correlation Analysis
# ─────────────────────────────────────────────────────────────
sc_t = StandardScaler(); T_sc = sc_t.fit_transform(temp_emb_all)
sc_g = StandardScaler(); G_sc = sc_g.fit_transform(graph_emb_all)

n_components = 5
cca = CCA(n_components=n_components, max_iter=1000)
cca.fit(T_sc, G_sc)
T_c, G_c = cca.transform(T_sc, G_sc)

canon_corrs = [np.corrcoef(T_c[:, i], G_c[:, i])[0, 1]
               for i in range(n_components)]

print("\nCanonical Correlations (temporal ↔ spatial):")
for i, r in enumerate(canon_corrs):
    print(f"  Component {i+1}: r = {r:.4f}")

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(range(1, n_components+1), canon_corrs,
       color='steelblue', edgecolor='black')
ax.axhline(0, color='black', lw=0.8)
ax.set_xlabel('Canonical Component'); ax.set_ylabel('Correlation  r')
ax.set_title('Stage 5 — Temporal ↔ Spatial CCA')
ax.set_ylim(-0.1, 1.0); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/stage5_cca_correlations.png', dpi=150)
plt.show()


# ─────────────────────────────────────────────────────────────
# CELL 33 — Per-class coupling scatter (1st canonical pair)
# ─────────────────────────────────────────────────────────────
color_map = np.array(['#E74C3C','#2ECC71','#3498DB'])

fig, axes = plt.subplots(1, N_CLASSES, figsize=(15, 4), sharey=True)
for cls, ax in enumerate(axes):
    mask = y_conn == cls
    ax.scatter(T_c[mask, 0], G_c[mask, 0],
               c=color_map[cls], alpha=0.4, s=8)
    r = np.corrcoef(T_c[mask, 0], G_c[mask, 0])[0, 1]
    ax.set_title(f'{CLASS_NAMES[cls]}  r={r:.3f}')
    ax.set_xlabel('Temporal CCA-1')
    ax.set_ylabel('Spatial CCA-1')
    ax.grid(alpha=0.3)

plt.suptitle('Stage 5 — Temporal–Spatial Coupling per Class', fontsize=12)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/stage5_coupling_scatter.png', dpi=150)
plt.show()
print("Stage 5 complete ✅")


# ╔═════════════════════════════════════════════════════════════╗
# ║                        STAGE  6                            ║
# ║         EARLY-WARNING / LEAD-TIME OPTIMISATION             ║
# ║  How few seconds of EEG are needed for diagnosis?          ║
# ╚═════════════════════════════════════════════════════════════╝

# ─────────────────────────────────────────────────────────────
# CELL 34 — Build subject-level segment index map
# ─────────────────────────────────────────────────────────────
def build_subject_index_map(labels_dict, y_conn):
    """
    Maps each subject to the contiguous block of segment indices
    that belong to it, in original recording order.
    Returns: {sub_id: (start_idx, end_idx, label_int)}
    """
    cls_to_int = {n: i for i, n in enumerate(CLASS_NAMES)}
    # Count segments per class that have been assigned so far
    class_seg_counts = {i: 0 for i in range(N_CLASSES)}
    # First pass: count total segments per class
    segs_per_class = {i: int((y_conn == i).sum()) for i in range(N_CLASSES)}
    # Build per-subject counts from the dataset builder order
    subject_map = {}
    # Re-run segment counting per subject (lightweight, no file I/O)
    ptr = 0
    for sub_id, label_str in sorted(labels_dict.items()):
        lbl = cls_to_int.get(label_str)
        if lbl is None: continue
        # Find consecutive run of this label starting at ptr
        start = ptr
        while ptr < len(y_conn) and y_conn[ptr] == lbl:
            ptr += 1
        if ptr > start:
            subject_map[sub_id] = (start, ptr, lbl)
    return subject_map

subject_map = build_subject_index_map(labels, y_conn)
print(f"Subject index map built: {len(subject_map)} subjects")


# ─────────────────────────────────────────────────────────────
# CELL 35 — Lead-time accuracy: ensemble of Stage 1 + Stage 3
# ─────────────────────────────────────────────────────────────
def lead_time_accuracy(seg_counts=(1, 2, 4, 8, 16, 32, 64)):
    """
    For each K in seg_counts, predict each subject using only
    their first K segments (majority vote, ensemble of CNN+GCN).
    Returns dict {K: balanced_accuracy}.
    """
    results = {}
    for K in seg_counts:
        preds, trues = [], []
        for sub_id, (start, end, lbl) in subject_map.items():
            idx = np.arange(start, min(start + K, end))
            if len(idx) == 0: continue

            # Stage 1 prediction
            p1 = model.predict(
                [X_raw[idx], X_band[idx]], verbose=0)      # (k, 3)
            # Stage 3 prediction
            p3 = gcn_model.predict(
                [X_node[idx], A_graph[idx]], verbose=0)    # (k, 3)

            # Ensemble: average softmax → majority vote
            p_ens = (p1 + p3) / 2.0
            pred  = Counter(np.argmax(p_ens, axis=1).tolist()).most_common(1)[0][0]
            preds.append(pred)
            trues.append(lbl)

        if preds:
            acc = balanced_accuracy_score(trues, preds)
            results[K] = acc
            print(f"  K={K:3d} segs  ({K*SEG_SEC:4d}s)  "
                  f"n_subjects={len(preds)}  bal-acc={acc:.4f}")
    return results

print("Running lead-time analysis …")
lt_results = lead_time_accuracy(seg_counts=[1, 2, 4, 8, 16, 32, 64])


# ─────────────────────────────────────────────────────────────
# CELL 36 — Plot lead-time accuracy curve
# ─────────────────────────────────────────────────────────────
seg_keys  = sorted(lt_results.keys())
acc_vals  = [lt_results[k] for k in seg_keys]
time_vals = [k * SEG_SEC for k in seg_keys]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(time_vals, acc_vals, marker='o', lw=2.5,
        color='darkorchid', markersize=8)
ax.axhline(1/3, color='gray', ls='--', lw=1.2, label='Random chance (33%)')
ax.fill_between(time_vals, 1/3, acc_vals, alpha=0.15, color='darkorchid')
ax.set_xlabel('EEG Recording Length Used  (seconds)')
ax.set_ylabel('Balanced Accuracy')
ax.set_title('Stage 6 — Early Warning: Lead-Time Accuracy\n'
             'How few seconds of EEG are needed for reliable diagnosis?')
ax.legend(); ax.grid(alpha=0.3); ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/stage6_lead_time.png', dpi=150)
plt.show()


# ─────────────────────────────────────────────────────────────
# CELL 37 — Entropy trajectory over recording time
# ─────────────────────────────────────────────────────────────
def entropy_over_time(A_all, y_conn, n_bins=10):
    N        = len(A_all)
    bin_size = N // n_bins
    results  = {cls: [] for cls in CLASS_NAMES}
    for b in range(n_bins):
        s = b * bin_size
        e = s + bin_size
        for cls_idx, cls_name in enumerate(CLASS_NAMES):
            mask = y_conn[s:e] == cls_idx
            if not mask.any():
                results[cls_name].append(np.nan); continue
            ents = []
            for A in A_all[s:e][mask]:
                A_nsl = A.copy(); np.fill_diagonal(A_nsl, 0)
                tri   = A_nsl[np.triu_indices(N_CH, k=1)]
                tri  /= (tri.sum() + 1e-12)
                ents.append(float(scipy_entropy(tri + 1e-12)))
            results[cls_name].append(float(np.mean(ents)))
    return results

print("Computing entropy trajectory …")
ent_traj = entropy_over_time(A_all, y_conn)

fig, ax = plt.subplots(figsize=(9, 4))
clr = {'AD':'#E74C3C','CN':'#2ECC71','FTD':'#3498DB'}
for cls, vals in ent_traj.items():
    ax.plot(range(len(vals)), vals, marker='o', lw=2,
            color=clr[cls], label=cls)

ax.set_xlabel('Temporal Bin (early → late recording)')
ax.set_ylabel('Mean Edge Entropy')
ax.set_title('Stage 6 — Entropy Trajectory Over Recording Time\n'
             'Rising entropy early → potential early-warning signal')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/stage6_entropy_trajectory.png', dpi=150)
plt.show()
print("Stage 6 complete ✅")


# ╔═════════════════════════════════════════════════════════════╗
# ║                    FINAL SUMMARY                           ║
# ╚═════════════════════════════════════════════════════════════╝

# ─────────────────────────────────────────────────────────────
# CELL 38 — Save all models + print summary
# ─────────────────────────────────────────────────────────────
gcn_model.save('./checkpoints/best_gcn.keras')
gcn_embed_model.save('./checkpoints/gcn_embed.keras')
temporal_extractor.save('./checkpoints/temporal_extractor.keras')

print("\n" + "="*60)
print("COMPLETE PIPELINE SUMMARY")
print("="*60)
print(f"""
Stage 1 — CNN-BiLSTM  (CELLS 2–18)
  Raw EEG → Conv1D × 3 → BiLSTM × 2 → channel attention
  Band-power branch (delta/theta/alpha/beta/gamma)
  Saved : ./checkpoints/best_model.keras
  Plots : training_curves, confusion_matrix, band_power,
          channel_attention

Stage 2 — Functional Connectivity  (CELLS 19–21)
  (19×19) coherence matrix per 4-second segment
  Saved : conn_cache.npz
  Plots : stage2_connectivity_matrices

Stage 3 — GCN Graph Learning  (CELLS 22–27)
  3-layer GCN  |  node=band-power  |  edge=coherence
  Global mean+max pool → 128-dim graph embedding
  Saved : ./checkpoints/best_gcn.keras
  Plots : stage3_gcn_results

Stage 4 — Connectivity Entropy Biomarker  (CELLS 28–30)
  VNE, edge entropy, strength entropy, global efficiency,
  clustering — Kruskal-Wallis + Mann-Whitney significance
  Saved : {OUT_DIR}/entropy_biomarkers.csv
  Plots : stage4_entropy_biomarkers

Stage 5 — Temporal–Spatial Coupling  (CELLS 31–33)
  CCA between Stage-1 (192-d) and Stage-3 (128-d) embeddings
  Plots : stage5_cca_correlations, stage5_coupling_scatter

Stage 6 — Early Warning / Lead-Time  (CELLS 34–37)
  Ensemble CNN+GCN majority-vote at K = 1…64 segments
  Entropy trajectory shows early-onset signal
  Plots : stage6_lead_time, stage6_entropy_trajectory

All outputs → {OUT_DIR}/
""")